# Information Capacity Signals in Linear-Recurrent LMs

**Research question.** For a fixed-size recurrent sequence model, *which information-theoretic
signal best reflects the information capacity demanded by the input data?*

We lay the study out as a **2-axis matrix**: `signal (Axis 1) × dataset (Axis 2)`, evaluated on
**two architectures** with a fixed-size recurrent state.

### Axis 1 — Signals (information-theoretic)
| id | Signal | What it captures |
|----|--------|------------------|
| **S1** | **Effective rank (eRank)** of the recurrent state | `exp(spectral entropy)` of the state's singular values → how many state directions are in use. |
| **S2** | **Predictive entropy / bits-per-token** | Shannon entropy of the next-token distribution + how fast in-context surprisal drops. |
| **S3** | **In-context epiplexity** | Area under the per-token NLL curve *above its asymptote* — the learnable structure the model extracts from the input (prequential code length, run in-context on the frozen model). arXiv:2601.03220. |
| **S4** | **Recall / decodable capacity** | For MQAR: effective bits stored vs. *known* ground-truth. Generally: how much of the input is linearly decodable from the state (linear probe). |

### Axis 2 — Data
| id | Dataset | Why |
|----|---------|-----|
| **D1** | **MQAR** (multi-query associative recall) | Synthetic k-v recall; ground-truth information content is *known* (`N·log2 |V|` bits) → **calibration anchor**. |
| **D2** | **Natural language** (WikiText / Pile) | Real, redundant, compressible data. |
| **D3** | **State-tracking** (A5/S5 word problems, parity, flip-flop) | Info must be *maintained* as evolving state. arXiv:2404.08819. |

### Models (fixed-size recurrent state)
- **`state-spaces/mamba2-370m`** — Mamba-2 SSD; state `(nheads, headdim, d_state)` per layer.
- **`linear-moe-hub/MoM-Gated-Deltanet-340M`** — Gated DeltaNet (flash-linear-attention); matrix-valued recurrent state per head.

> **Scope of this notebook (v0):** all sections are scaffolded. The **worked example** (Section 7) fully
> implements **S1 · eRank on D1 · MQAR with mamba2-370m** end-to-end. Signal functions S2/S3/S4 are
> implemented as helpers; the full `signal × dataset × model` sweep (Section 8), D2/D3 builders, the GDN
> state extractor, and the S4 linear probe are marked **TODO** for the next pass.


## 0. Setup

Run in a CUDA environment. Uncomment to install.

In [ ]:
# !pip install torch transformers datasets
# !pip install mamba-ssm causal-conv1d          # for state-spaces/mamba2-370m
# !pip install flash-linear-attention            # for the Gated-DeltaNet model (fla)
# !pip install matplotlib seaborn scipy scikit-learn tqdm

## 1. Imports & config

In [ ]:
import os, sys, math, json, warnings
from collections import defaultdict
import numpy as np
import torch
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# Reuse the repo's helpers (effective_rank, get_ssm_states, ...) from mamba3_analysis/utils.py
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))   # notebooks/ -> repo root
sys.path.insert(0, os.path.join(REPO_ROOT, "mamba3_analysis"))
try:
    import utils as repo_utils          # effective_rank, get_ssm_states, get_A_disc, ...
    print("Imported repo utils:", [f for f in dir(repo_utils) if not f.startswith("_")][:8])
except Exception as e:
    repo_utils = None
    print("repo utils not importable (fine, we redefine what we need):", e)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)
print("Device:", DEVICE)

In [ ]:
# Central registries -------------------------------------------------------
MODELS = {
    "mamba2-370m": {"hf": "state-spaces/mamba2-370m", "kind": "mamba2"},
    "gdn-340m":    {"hf": "linear-moe-hub/MoM-Gated-Deltanet-340M", "kind": "gdn"},
}
SIGNALS  = ["S1_erank", "S2_pred_entropy", "S3_epiplexity", "S4_recall_capacity"]
DATASETS = ["D1_mqar", "D2_natural_language", "D3_state_tracking"]
RESULTS_DIR = os.path.join(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd())=="notebooks" else os.getcwd(),
                           "notebooks", "capacity_results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("results ->", RESULTS_DIR)

## 2. Model loading & unified state interface

Each backend exposes the same interface via a `ModelBundle`:
- `bundle.logits(input_ids)` -> `(B, T, V)` logits (for S2 / S3)
- `bundle.states(input_ids)` -> `{layer_idx: state_tensor}` final recurrent state (for S1 / S4)


In [ ]:
class ModelBundle:
    def __init__(self, name, model, tokenizer, kind):
        self.name, self.model, self.tokenizer, self.kind = name, model, tokenizer, kind

    @torch.no_grad()
    def logits(self, input_ids):
        input_ids = input_ids.to(DEVICE)
        out = self.model(input_ids)
        return out.logits if hasattr(out, "logits") else out[0]

    @torch.no_grad()
    def states(self, input_ids):
        input_ids = input_ids.to(DEVICE)
        if self.kind == "mamba2":
            return _mamba2_states(self.model, input_ids)
        elif self.kind == "gdn":
            return _gdn_states(self.model, input_ids)   # TODO: verify against installed fla
        raise ValueError(self.kind)


def load_bundle(name):
    spec = MODELS[name]
    if spec["kind"] == "mamba2":
        from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")
        model = MambaLMHeadModel.from_pretrained(spec["hf"], device=DEVICE, dtype=torch.float32)
        model.eval()
        return ModelBundle(name, model, tok, "mamba2")
    elif spec["kind"] == "gdn":
        # MoM Gated-DeltaNet ships with NO custom modeling code. The architecture
        # (model_type='mom_gated_deltanet') is registered with transformers simply by
        # importing fla, so AutoModel can build it -- trust_remote_code is NOT needed.
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import fla  # noqa: F401  registers mom_gated_deltanet with transformers
        tok = AutoTokenizer.from_pretrained(spec["hf"])
        model = AutoModelForCausalLM.from_pretrained(spec["hf"], torch_dtype=torch.float32).to(DEVICE)
        model.eval()
        return ModelBundle(name, model, tok, "gdn")
    raise ValueError(name)

### 2a. State extraction — Mamba-2 (works today)
Uses `InferenceParams`; final SSM state per layer has shape `(nheads, headdim, d_state)`
(mamba2-370m: `24 × 64 × 128`). Mirrors `mamba3_analysis/utils.get_ssm_states`.

In [ ]:
def _mamba2_states(model, input_ids):
    from mamba_ssm.utils.generation import InferenceParams
    if input_ids.dim() == 1:
        input_ids = input_ids.unsqueeze(0)
    ip = InferenceParams(max_seqlen=input_ids.shape[1], max_batch_size=input_ids.shape[0])
    with torch.no_grad():
        _ = model(input_ids, inference_params=ip)
    states = {}
    for layer_idx, (conv_state, ssm_state) in ip.key_value_memory_dict.items():
        # ssm_state: (batch, nheads, headdim, d_state) -> drop batch (assume B=1 here)
        states[layer_idx] = ssm_state[0].detach().cpu().float()
    return states

### 2b. State extraction — MoM Gated-DeltaNet  *(implemented; verify layout on GPU via self-check)*

`config`: `model_type='mom_gated_deltanet'`, 24 layers, `num_heads=4`, `head_dim=256`, `expand_v=1`,
**`num_memories=4`**, `topk=2`. No custom remote code — the arch is registered by `import fla`.

MoM keeps a **mixture of memories**, so the cache is *not* a single matrix like Mamba-2. Via the fla
`Cache` API (`transformers>4.53.3` → `FLACache`, else `LegacyFLACache`), `cache[layer]['recurrent_state']`
is a **list** `[primary, (shared?)]`. The primary memory stacks the `num_memories` states along the
batch axis → `(batch·num_memories, num_heads, head_k, head_v)`. The delta-rule recurrent state puts the
matrix in the **last two dims** `(head_k, head_v)`.

Our extractor assumes **batch = 1** (call `states()` on one sequence) and returns, per layer, a stack of
`num_memories · num_heads` matrices of shape `(head_k, head_v)` — so `S1_erank` treats each *memory-head*
as one unit (memory axis flattened in). Run the **self-check cell** below on your GPU box to confirm the
layout for your installed `fla` version before trusting the numbers.

In [ ]:
def _gdn_states(model, input_ids):
    """Final recurrent state per layer for MoM Gated-DeltaNet (flash-linear-attention).

    Returns {layer_idx: Tensor(n_matrices, head_k, head_v)} with n_matrices =
    num_memories * num_heads (+ shared memory if present), assuming batch size 1.
    The matrix is taken from the last two dims of whatever tensors the cache holds,
    so this is robust to leading-axis layout (batch / memory / head ordering).
    """
    if input_ids.dim() == 1:
        input_ids = input_ids.unsqueeze(0)
    assert input_ids.shape[0] == 1, "_gdn_states assumes batch size 1"
    with torch.no_grad():
        out = model(input_ids, use_cache=True)
    cache = out.past_key_values
    n_layers = getattr(model.config, "num_hidden_layers", None) or len(cache)
    states = {}
    for i in range(n_layers):
        try:
            rs = cache[i]["recurrent_state"]        # FLACache/LegacyFLACache -> layer state dict
        except Exception:
            continue
        elems = rs if isinstance(rs, (list, tuple)) else [rs]   # MoM: [primary, (shared?)]
        mats = []
        for el in elems:
            if el is None or not torch.is_tensor(el) or el.dim() < 2:
                continue
            t = el.detach().cpu().float()
            mats.append(t.reshape(-1, t.shape[-2], t.shape[-1]))   # (*, head_k, head_v)
        if mats:
            states[i] = torch.cat(mats, dim=0)      # (num_memories*num_heads[+shared], head_k, head_v)
    if not states:
        raise RuntimeError("No recurrent_state found in cache — run the 2b self-check to inspect layout.")
    return states

#### 2b self-check *(safe to run top-to-bottom — skips cleanly if `fla`/GPU/model are unavailable)*
Prints the live cache layout and exercises the extractor, so you can confirm the MoM memory axis is
being flattened as intended.

In [ ]:
def gdn_state_selfcheck(seq_len=32):
    _b = load_bundle("gdn-340m")
    cfg = _b.model.config
    print(f"model_type={cfg.model_type} layers={cfg.num_hidden_layers} "
          f"num_heads={getattr(cfg,'num_heads','?')} head_dim={getattr(cfg,'head_dim','?')} "
          f"num_memories={getattr(cfg,'num_memories','?')}")
    _ids = torch.randint(0, 1000, (1, seq_len)).to(DEVICE)
    with torch.no_grad():
        _out = _b.model(_ids, use_cache=True)
    _cache = _out.past_key_values
    print("cache class:", type(_cache).__name__)
    _rs = _cache[0]["recurrent_state"]
    _elems = _rs if isinstance(_rs, (list, tuple)) else [_rs]
    print("layer0 recurrent_state:", type(_rs).__name__,
          "elems ->", [None if e is None else tuple(e.shape) for e in _elems])
    _st = _b.states(_ids)
    print("extractor layer0 ->", tuple(_st[0].shape), "= (num_memories*num_heads, head_k, head_v)")
    return _b

try:
    gdn_state_selfcheck()
except Exception as e:
    print("GDN self-check skipped:", type(e).__name__, "-", str(e)[:140])

## 3. Axis 1 — Signal implementations

All signals reduce to a single scalar per (sequence, model) so they can populate the matrix.

### S1 · Effective rank of the recurrent state  *(fully implemented — used in the worked example)*
`eRank(M) = exp(H)` where `H` is the Shannon entropy of the normalized singular-value spectrum of `M`.
We compute it per matrix and average. For Mamba-2 each matrix is a head's `(headdim, d_state)`; for MoM
Gated-DeltaNet each matrix is a *memory-head*'s `(head_k, head_v)` (the extractor flattens the
`num_memories × num_heads` axes), so the same code covers both.

In [ ]:
def effective_rank(matrix):
    """exp(Shannon entropy of normalized singular values). Matches repo_utils.effective_rank."""
    if isinstance(matrix, torch.Tensor):
        matrix = matrix.cpu().numpy()
    s = np.linalg.svd(matrix, compute_uv=False)
    s = s / (s.sum() + 1e-12)
    return float(np.exp(-np.sum(s * np.log(s + 1e-12))))


def S1_erank(bundle, input_ids):
    """Mean per-head effective rank of the final recurrent state, averaged over layers."""
    states = bundle.states(input_ids)          # {layer: (nheads, headdim, d_state)}
    per_layer = []
    for layer_idx, st in states.items():
        if st.dim() == 2:                       # flat state -> single matrix
            per_layer.append(effective_rank(st))
        else:                                   # (nheads, d1, d2) -> mean over heads
            per_layer.append(np.mean([effective_rank(st[h]) for h in range(st.shape[0])]))
    return {"erank_mean": float(np.mean(per_layer)),
            "erank_per_layer": np.asarray(per_layer)}

### S2 · Predictive entropy / bits-per-token  *(implemented helper)*
Shannon entropy of the next-token distribution (nats), plus mean bits-per-token (NLL of the realized
next token). Low predictive entropy late in a sequence ⇒ the state has absorbed usable structure.

In [ ]:
@torch.no_grad()
def S2_pred_entropy(bundle, input_ids):
    if input_ids.dim() == 1: input_ids = input_ids.unsqueeze(0)
    logits = bundle.logits(input_ids)                       # (B,T,V)
    logp = torch.log_softmax(logits.float(), dim=-1)
    p = logp.exp()
    ent = -(p * logp).sum(-1)                               # (B,T) predictive entropy, nats
    # realized bits-per-token: NLL of the true next token
    tgt = input_ids[:, 1:]
    nll = -logp[:, :-1, :].gather(-1, tgt.unsqueeze(-1)).squeeze(-1)   # (B,T-1) nats
    return {"pred_entropy_mean": float(ent.mean()),
            "bits_per_token": float(nll.mean() / math.log(2)),
            "pred_entropy_curve": ent[0].cpu().numpy()}

### S3 · In-context epiplexity  *(implemented helper)*
Prequential code length run **in-context** on the frozen model: the area under the per-token NLL
curve *above its asymptote*. Let `nll_t = -log p(x_t | x_{<t})` and let `A` = asymptotic loss (mean
NLL over the final fraction of the sequence). Then
`epiplexity ≈ Σ_t max(nll_t − A, 0)` — the extra bits paid early, i.e. the learnable structure the
model extracts as context accumulates (arXiv:2601.03220, prequential estimator).

In [ ]:
@torch.no_grad()
def S3_epiplexity(bundle, input_ids, tail_frac=0.2):
    if input_ids.dim() == 1: input_ids = input_ids.unsqueeze(0)
    logits = bundle.logits(input_ids)
    logp = torch.log_softmax(logits.float(), dim=-1)
    tgt = input_ids[:, 1:]
    nll = -logp[:, :-1, :].gather(-1, tgt.unsqueeze(-1)).squeeze(-1)[0]  # (T-1,) nats
    nll_bits = nll.cpu().numpy() / math.log(2)
    k = max(1, int(len(nll_bits) * tail_frac))
    asymptote = float(nll_bits[-k:].mean())                 # final loss
    excess = np.clip(nll_bits - asymptote, 0, None)
    return {"epiplexity_bits": float(excess.sum()),         # area above final loss
            "asymptote_bits": asymptote,
            "nll_curve_bits": nll_bits}

### S4 · Recall / decodable capacity  **[TODO: linear probe]**
Two flavors: (a) *ground-truth* bits for MQAR = `N · log2|V|` (known); (b) *decodable* bits = how well
a linear probe recovers stored values from the final state. Ground-truth is implemented; the probe is
a TODO next-pass.

In [ ]:
def S4_ground_truth_bits(n_pairs, vocab):
    """Known information content of an MQAR instance, in bits."""
    return float(n_pairs * math.log2(vocab))


def S4_recall_capacity(bundle, input_ids, meta=None):
    # TODO(next-pass): fit a linear probe from bundle.states(input_ids) -> stored values,
    #   report decodable bits = accuracy-weighted N*log2|V|. For now return ground truth if known.
    gt = S4_ground_truth_bits(meta["n_pairs"], meta["vocab"]) if meta else None
    return {"ground_truth_bits": gt, "decodable_bits": None}  # decodable_bits: TODO

## 4. Axis 2 — Dataset builders

### D1 · MQAR  *(fully implemented — used in the worked example)*
We build sequences **directly as token ids** (MQAR is synthetic, so we bypass the tokenizer): sample
`N` distinct keys and their values from a reserved id range, interleave `k v k v ...`, then append the
keys again as queries. Increasing `N` increases the true information load `N·log2|V|`.

In [ ]:
def make_mqar_batch(n_pairs, vocab=512, key_offset=1000, val_offset=2000,
                    n_seq=8, query_all=True, seed=0):
    """Return (input_ids [n_seq, T], meta). Ids kept well inside the model vocab (~50k)."""
    rng = np.random.default_rng(seed)
    seqs = []
    for _ in range(n_seq):
        keys = rng.choice(vocab, size=n_pairs, replace=False) + key_offset
        vals = rng.choice(vocab, size=n_pairs, replace=True)  + val_offset
        pairs = np.empty(n_pairs * 2, dtype=np.int64)
        pairs[0::2] = keys; pairs[1::2] = vals
        queries = keys if query_all else keys[: max(1, n_pairs // 2)]
        seqs.append(np.concatenate([pairs, queries]))
    T = min(len(s) for s in seqs)
    ids = torch.tensor(np.stack([s[:T] for s in seqs]), dtype=torch.long)
    meta = {"n_pairs": n_pairs, "vocab": vocab, "T": T,
            "gt_bits": S4_ground_truth_bits(n_pairs, vocab)}
    return ids, meta

### D2 · Natural language  **[TODO: full loader]**
Skeleton for WikiText-103 windows. Fill in tokenization + windowing next pass.

In [ ]:
def make_natural_language_batch(seq_len=512, n_seq=8, split="test", seed=0):
    # TODO(next-pass):
    #   from datasets import load_dataset
    #   ds = load_dataset("wikitext", "wikitext-103-raw-v1", split=split)
    #   tok = load_bundle("mamba2-370m").tokenizer  (or pass tokenizer in)
    #   tokenize, chunk into n_seq windows of seq_len, return input_ids
    raise NotImplementedError("D2 natural-language loader — TODO")

### D3 · State-tracking  **[TODO: generators]**
Word problems on the alternating group A5 (NC1-complete; a minimal *hard* state-tracking benchmark),
plus parity and flip-flop as easy references. Target = the running product / current state; the label
sequence forces the model to *maintain* state.

In [ ]:
def make_state_tracking_batch(task="A5", length=64, n_seq=8, seed=0):
    # TODO(next-pass): emit sequences of group elements (A5/S5) or parity/flip-flop instructions,
    #   with targets = running product / current bit. Encode as token ids the model can read.
    #   Refs: Merrill et al. 'Illusion of State' (2404.08819); DeltaProduct (2502.10297).
    raise NotImplementedError(f"D3 state-tracking generator '{task}' — TODO")

DATASET_BUILDERS = {
    "D1_mqar": make_mqar_batch,
    "D2_natural_language": make_natural_language_batch,
    "D3_state_tracking": make_state_tracking_batch,
}

## 5. WORKED EXAMPLE — S1 · eRank on D1 · MQAR (both models, side by side)

Fully runnable (needs GPU + `mamba-ssm` for mamba2-370m and `flash-linear-attention` for the MoM
Gated-DeltaNet). We sweep the number of key-value pairs `N` (the information load) and measure the mean
effective rank of the final recurrent state for **both** models.

Because the two architectures store different-sized state matrices — Mamba-2 `64×128` → max rank 64;
MoM Gated-DeltaNet `256×256` → max rank 256 — raw eRank is not directly comparable, so we also plot
**normalized eRank** (`eRank / full rank`) as a fair *capacity-utilization* measure. Expectation:
eRank rises with `N` until the fixed-size state **saturates** — that saturation point is the capacity
signal we are hunting for.

In [ ]:
# 5.1 Load both models (downloads weights on first run)
MODELS_TO_RUN = ["mamba2-370m", "gdn-340m"]
BUNDLES = {}
for _name in MODELS_TO_RUN:
    try:
        BUNDLES[_name] = load_bundle(_name)
        print("loaded", _name)
    except Exception as e:
        print("could NOT load", _name, "->", type(e).__name__, str(e)[:140])

In [ ]:
# 5.2 Sweep N and measure eRank for each model
N_GRID = [2, 4, 8, 16, 32, 64, 128]
VOCAB  = 512
sweep, maxrank = {}, {}
for _name, b in BUNDLES.items():
    # infer max possible rank = min(d1, d2) of the state matrices, for normalization
    ids0, _ = make_mqar_batch(n_pairs=8, vocab=VOCAB, n_seq=1, seed=0)
    shp = next(iter(b.states(ids0[:1]).values())).shape
    maxrank[_name] = int(min(shp[-2], shp[-1]))
    rows = []
    for n in N_GRID:
        ids, meta = make_mqar_batch(n_pairs=n, vocab=VOCAB, n_seq=4, seed=0)
        er = [S1_erank(b, ids[i:i+1])["erank_mean"] for i in range(ids.shape[0])]
        rows.append({"n_pairs": n, "T": meta["T"], "gt_bits": meta["gt_bits"],
                     "erank_mean": float(np.mean(er)), "erank_std": float(np.std(er))})
        print(f"{_name:12s} N={n:>3} T={meta['T']:>4} eRank={np.mean(er):6.3f} (max {maxrank[_name]})")
    sweep[_name] = rows

with open(os.path.join(RESULTS_DIR, "worked_example_S1_D1_both.json"), "w") as f:
    json.dump({"sweep": sweep, "maxrank": maxrank}, f, indent=2)

In [ ]:
# 5.3 Compare the two models: raw eRank and normalized eRank vs load
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for _name, rows in sweep.items():
    N  = [r["n_pairs"] for r in rows]
    er = np.array([r["erank_mean"] for r in rows])
    es = np.array([r["erank_std"] for r in rows])
    mr = maxrank[_name]
    ax[0].errorbar(N, er, yerr=es, marker="o", capsize=3, label=_name)
    ax[1].errorbar(N, er / mr, yerr=es / mr, marker="o", capsize=3, label=f"{_name} (/{mr})")
ax[0].set_ylabel("mean eRank of recurrent state"); ax[0].set_title("S1 eRank vs MQAR load")
ax[1].set_ylabel("normalized eRank (fraction of full rank)")
ax[1].set_title("capacity utilization (fair cross-arch)")
for a in ax:
    a.set_xscale("log", base=2); a.set_xlabel("# key-value pairs N"); a.grid(alpha=.3); a.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "worked_example_S1_D1_both.png"), dpi=120)
plt.show()
print("eRank saturation as N grows = capacity signal; the right panel compares the two "
      "architectures fairly (normalized).")

## 6. Full experiment matrix  **[TODO: sweep]**

Once D2/D3 builders, the GDN extractor, and the S4 probe are filled in, run every
`signal × dataset × model` cell and collect scalars into a tidy table + heatmap. The analysis we care
about: **which signal is most monotonic with the known MQAR capacity, and most discriminative across
D1/D2/D3** — that signal is the answer to the research question.

In [ ]:
SIGNAL_FNS = {
    "S1_erank": S1_erank,
    "S2_pred_entropy": S2_pred_entropy,
    "S3_epiplexity": S3_epiplexity,
    "S4_recall_capacity": S4_recall_capacity,   # needs meta for ground truth
}

def run_matrix(models=("mamba2-370m",), datasets=("D1_mqar",),
               signals=("S1_erank","S2_pred_entropy","S3_epiplexity"), n_pairs=16):
    """TODO(next-pass): generalize dataset args; handle D2/D3; aggregate to DataFrame + heatmap."""
    records = []
    for mname in models:
        bundle = load_bundle(mname)
        for dname in datasets:
            if dname == "D1_mqar":
                ids, meta = make_mqar_batch(n_pairs=n_pairs)
            else:
                raise NotImplementedError(f"{dname} not wired yet")   # TODO
            for sname in signals:
                fn = SIGNAL_FNS[sname]
                out = fn(bundle, ids[:1], meta) if sname == "S4_recall_capacity" else fn(bundle, ids[:1])
                scalar = next((out[k] for k in
                    ("erank_mean","pred_entropy_mean","epiplexity_bits","ground_truth_bits") if k in out), None)
                records.append({"model": mname, "dataset": dname, "signal": sname, "value": scalar})
    return records

# records = run_matrix(); records

## 7. Analysis — which signal best tracks capacity?  **[TODO]**

Plan for the analysis pass:
1. **Calibration on D1 (MQAR).** For each signal, correlate its value with `gt_bits = N·log2|V|`
   across the N-sweep (Spearman ρ). The signal that tracks ground-truth capacity best wins the
   calibration.
2. **Discriminability across D1/D2/D3.** Does the signal separate the three data regimes
   (recall vs. compressible NL vs. state-tracking)? Report effect sizes.
3. **Architecture check.** Repeat for `gdn-340m`; does the best signal agree across Mamba-2 vs
   Gated DeltaNet, or is it architecture-specific?
4. **Deliverable.** A `signal × dataset` heatmap + a ranking of signals by (calibration ρ ×
   discriminability), naming the single best "information-capacity" signal.
